In [ ]:
import os

from pathlib import Path

BASE_PATH = Path.cwd().parent.parent

RAW = BASE_PATH / 'Data' / 'raw'
TEMP = BASE_PATH / 'Data' / 'temp'
USE = BASE_PATH / 'Data' / 'use'
FIGURES = BASE_PATH / 'Results' / 'Figures'
TABLES = BASE_PATH / 'Results' / 'Tables'

for path in [RAW, TEMP, USE, FIGURES, TABLES]:
    path.mkdir(parents=True, exist_ok=True)

print(f"✅ BASE_PATH: {BASE_PATH}")

In [ ]:
# larger v.s. small scale data center: figure 3c data preparation


import os
import pandas as pd
import numpy as np
from shapely.geometry import Point
import geopandas as gpd
from tqdm import tqdm
import time
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

def calculate_distance_vectorized(lat1, lon1, lat2_array, lon2_array):
    """
    Vectorized distance calculation between two points (Haversine formula)
    
    Parameters:
        lat1, lon1: latitude and longitude of one point
        lat2_array, lon2_array: arrays of latitudes and longitudes of multiple points
    
    Returns:
        Distance array (unit: km)
    """
    # Convert to radians
    lat1_rad = np.radians(lat1)
    lon1_rad = np.radians(lon1)
    lat2_rad = np.radians(lat2_array)
    lon2_rad = np.radians(lon2_array)
    
    # Haversine formula
    dlat = lat2_rad - lat1_rad
    dlon = lon2_rad - lon1_rad
    
    a = np.sin(dlat/2)**2 + np.cos(lat1_rad) * np.cos(lat2_rad) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    
    # Earth radius (km)
    r = 6371
    
    return c * r

def match_gadm_attributes(df_coal, gadm_path):
    """
    Match GADM administrative attributes by plant coordinates
    
    Parameters:
        df_coal: DataFrame (with Latitude and Longitude)
        gadm_path: path to GADM gpkg file
    
    Returns:
        Matched DataFrame
    """
    print("\n🗺️  Matching GADM administrative attributes...")
    
    # Read GADM data
    try:
        gadm_gdf = gpd.read_file(gadm_path)
        print(f"  ✓ GADM data loaded: {len(gadm_gdf)} records")
        print(f"  ✓ GADM fields: {list(gadm_gdf.columns)}")
    except Exception as e:
        print(f"  ❌ Failed to read GADM: {e}")
        return df_coal
    
    # Create GeoDataFrame using plant coordinates
    geometry = [Point(xy) for xy in zip(df_coal['Longitude'], df_coal['Latitude'])]
    coal_gdf = gpd.GeoDataFrame(df_coal, geometry=geometry, crs='EPSG:4326')
    
    # Ensure CRS consistency
    if gadm_gdf.crs != coal_gdf.crs:
        print(f"  - Convert GADM CRS: {gadm_gdf.crs} → {coal_gdf.crs}")
        gadm_gdf = gadm_gdf.to_crs(coal_gdf.crs)
    
    # Spatial join
    print("  - Running spatial join (may take a few minutes)...")
    start_time = time.time()
    
    coal_matched = gpd.sjoin(coal_gdf, gadm_gdf, how='left', predicate='within')
    
    elapsed = time.time() - start_time
    print(f"  ✓ Spatial join done, time: {elapsed/60:.1f} min")
    
    # Match summary
    matched_count = coal_matched['index_right'].notna().sum()
    print(f"  ✓ Matched: {matched_count:,}/{len(df_coal):,} records ({matched_count/len(df_coal)*100:.1f}%)")
    
    # Get GADM attribute fields (exclude geometry and index_right)
    gadm_cols = [col for col in coal_matched.columns 
                 if col not in df_coal.columns 
                 and col not in ['geometry', 'index_right']]
    
    if gadm_cols:
        print(f"  ✓ Matched GADM fields ({len(gadm_cols)}): {gadm_cols[:10]}")
        if len(gadm_cols) > 10:
            print(f"    ... {len(gadm_cols)-10} more fields")
    
    # Drop extra columns created by spatial join
    cols_to_drop = ['geometry', 'index_right']
    coal_matched = coal_matched.drop(columns=[c for c in cols_to_drop if c in coal_matched.columns])
    
    # Convert back to regular DataFrame
    coal_matched = pd.DataFrame(coal_matched)
    
    return coal_matched

def process_proximity_exposure():
    """
    Main function: calculate inverse-distance-based exposure metrics
    (multiple time windows + full type decomposition)
    """
    
    # ===== 1. Read data =====
    coal_plants_path = os.path.join(TEMP, "gem_coal_plants_multi_record_sa_sinceoperating.dta")
    df_coal = pd.read_stata(coal_plants_path)
    
    ai_center_path = os.path.join(RAW, "SPGlobal_Export.xlsx")
    df_ai = pd.read_excel(ai_center_path, sheet_name='Sheet1')
    
    gadm_path = os.path.join(RAW, "gadm_410.gpkg")
    
    # ===== 1.5 Match GADM attributes =====
    df_coal = match_gadm_attributes(df_coal, gadm_path)
    
    # ===== 2. Data preprocessing =====
    print("\n🔧 Step 2/6: Data preprocessing...")
    
    # Process AI center built year
    df_ai['YR_BUILT'] = pd.to_numeric(df_ai['YR_BUILT'], errors='coerce')
    
    # Check SECONDARY_PPTY_TYPE field
    if 'SECONDARY_PPTY_TYPE' in df_ai.columns:
        print(f"  ✓ SECONDARY_PPTY_TYPE field found")
        print(f"  ✓ Type distribution:")
        type_counts = df_ai['SECONDARY_PPTY_TYPE'].value_counts()
        for ptype, count in type_counts.head(10).items():
            print(f"    - {ptype}: {count}")
        if len(type_counts) > 10:
            print(f"    ... ({len(type_counts)} types in total)")
    else:
        print(f"  ⚠️  SECONDARY_PPTY_TYPE not found, all AI centers will be treated as 'other'")
        df_ai['SECONDARY_PPTY_TYPE'] = 'other'
    
    # Drop missing values
    df_ai_filtered = df_ai.dropna(subset=['LATITUDE', 'LONGITUDE', 'YR_BUILT']).copy()
    print(f"  ✓ Valid AI centers: {len(df_ai_filtered):,}")
    
    df_coal = df_coal.dropna(subset=['Latitude', 'Longitude', 'year'])
    print(f"  ✓ Valid coal plant records: {len(df_coal):,}")
    
    # Convert to numpy arrays
    ai_lat = df_ai_filtered['LATITUDE'].values
    ai_lon = df_ai_filtered['LONGITUDE'].values
    ai_year = df_ai_filtered['YR_BUILT'].values.astype(int)
    
    # Create type indicator array (hyperscale vs other)
    ai_type = df_ai_filtered['SECONDARY_PPTY_TYPE'].fillna('other').values
    
    # Define hyperscale types, 'Wholesale Data Center'?
    hyperscale_types = ['Hyperscale Data Center', 'Crypto Mining Data Center', 'Cloud Data Center', 'Wholesale Data Center']
    ai_is_hyperscale = np.isin(ai_type, hyperscale_types)
    
    hyperscale_count = ai_is_hyperscale.sum()
    other_count = (~ai_is_hyperscale).sum()
    
    print(f"\n  AI center type summary:")
    print(f"    - Hyperscale (incl. Crypto Mining): {hyperscale_count:,} ({hyperscale_count/len(df_ai_filtered)*100:.1f}%)")
    print(f"    - Other: {other_count:,} ({other_count/len(df_ai_filtered)*100:.1f}%)")
    
    print(f"\n  ✓ AI center year range: {ai_year.min()} - {ai_year.max()}")
    
    # Show year distribution
    year_counts = pd.Series(ai_year).value_counts().sort_index()
    print(f"  ✓ AI center year distribution (first 10 years):")
    for year, count in year_counts.head(10).items():
        print(f"    - {year}: {count}")
    if len(year_counts) > 10:
        print(f"    ... ({len(year_counts)} years in total)")
    
    # ===== 3. Define parameters =====
    print("\n📊 Step 3/6: Define exposure parameters...")
    
    buffer_distances = [15, 25, 50]  # 15km, 25km, 50km (unit: km)
    
    # Four time windows
    time_windows = {
        'before06': (None, 2005),      # 2005 and earlier
        '06_15': (2006, 2015),          # 2006-2015
        '16_19': (2016, 2019),          # 2016-2019
        '20_24': (2020, 2024)           # 2020-2024
    }
    
    print(f"  ✓ Buffer radii: {[f'{d}km' for d in buffer_distances]}")
    print(f"  ✓ Time windows:")
    for window_name, (start, end) in time_windows.items():
        if start is None:
            print(f"    • {window_name}: ≤{end}")
        else:
            print(f"    • {window_name}: {start}-{end}")
    
    # Calculate total variables
    base_vars = len(buffer_distances) * len(time_windows)  # Base variables
    type_vars = len(buffer_distances) * len(time_windows) * 2  # Full type decomposition variables (hyper + other) for all time windows
    total_vars = base_vars + type_vars
    
    print(f"  ✓ Total variables: {base_vars} (base) + {type_vars} (type decomposition) = {total_vars}")
    
    # ===== 4. Create new variables =====
    print("\n🎯 Step 4/6: Create exposure variables...")
    
    new_columns = []
    
    # 4.1 Base variables (all time windows)
    for distance_km in buffer_distances:
        for window_name in time_windows.keys():
            col_name = f'ai_proximity_{distance_km}km_{window_name}'
            df_coal[col_name] = 0.0
            new_columns.append(col_name)
    
    # 4.2 Type decomposition variables (all time windows)
    for distance_km in buffer_distances:
        for window_name in time_windows.keys():
            # Hyperscale type
            col_name_hyper = f'ai_proximity_{distance_km}km_{window_name}_hyper'
            df_coal[col_name_hyper] = 0.0
            new_columns.append(col_name_hyper)
            
            # Other type
            col_name_other = f'ai_proximity_{distance_km}km_{window_name}_other'
            df_coal[col_name_other] = 0.0
            new_columns.append(col_name_other)
    
    print(f"  ✓ Created {len(new_columns)} exposure variables:")
    print(f"\n  📍 Base variables ({base_vars}):")
    for i, col in enumerate([c for c in new_columns if not c.endswith('_hyper') and not c.endswith('_other')], 1):
        print(f"    {i:2d}. {col}")
    
    print(f"\n  🆕 Type decomposition variables ({type_vars}):")
    for i, col in enumerate([c for c in new_columns if c.endswith('_hyper') or c.endswith('_other')], 1):
        print(f"    {i:2d}. {col}")
    
    # ===== 5. Calculate exposure metrics (core calculation) =====
    print("\n🚀 Step 5/6: Calculate inverse-distance-weighted exposure metrics...")
    print(f"  - Formula: PE = Σ(1/distance_km) for AI centers built in the time window within the buffer")
    print(f"  - Distance unit: km")
    print(f"  - Type decomposition:")
    print(f"    • hyper: Hyperscale Data Center + Crypto Mining Data Center + Wholesale + Cloud")
    print(f"    • other: all other types")
    print(f"  - Applied to all time windows: before06, 06_15, 16_19, 20_24")
    print(f"  - Total calculations: {len(df_coal):,} records")
    
    start_time = time.time()
    processed_count = 0
    last_report_time = start_time
    
    # Process by GEM_unit_phase_ID group
    grouped = df_coal.groupby('GEM_unit_phase_ID')
    
    for gem_id, group_df in tqdm(grouped, desc="Processing coal plants", ncols=80):
        # Get coordinates of this coal plant
        coal_lat = group_df['Latitude'].iloc[0]
        coal_lon = group_df['Longitude'].iloc[0]
        
        # Vectorized calculation of distances from this coal plant to all AI centers (km)
        distances_km = calculate_distance_vectorized(coal_lat, coal_lon, ai_lat, ai_lon)
        
        # Calculate exposure metrics for each year of this coal plant
        for idx, row in group_df.iterrows():
            current_year = int(row['year'])
            
            # For each buffer radius
            for distance_km in buffer_distances:
                
                # Find AI centers within the buffer
                within_buffer = distances_km <= distance_km
                
                # ===== 5.1 Base variables + type decomposition variables (all time windows) =====
                for window_name, (start_year, end_year) in time_windows.items():
                    
                    # 5.1.1 Base variable
                    col_name = f'ai_proximity_{distance_km}km_{window_name}'
                    
                    # Filter conditions
                    if start_year is None:
                        valid_ai = within_buffer & (ai_year <= end_year) & (ai_year <= current_year)
                    else:
                        valid_ai = within_buffer & (ai_year >= start_year) & (ai_year <= end_year) & (ai_year <= current_year)
                    
                    if valid_ai.sum() > 0:
                        valid_distances_km = distances_km[valid_ai]
                        valid_distances_km = np.maximum(valid_distances_km, 0.1)
                        valid_distances_10km = valid_distances_km / 10.0
                        proximity_exposure = np.sum(1.0 / valid_distances_10km)
                        df_coal.at[idx, col_name] = proximity_exposure
                    else:
                        df_coal.at[idx, col_name] = 0.0
                    
                    # 5.1.2 Type decomposition variables
                    # Hyperscale type
                    col_name_hyper = f'ai_proximity_{distance_km}km_{window_name}_hyper'
                    valid_ai_hyper = valid_ai & ai_is_hyperscale
                    
                    if valid_ai_hyper.sum() > 0:
                        valid_distances_km = distances_km[valid_ai_hyper]
                        valid_distances_km = np.maximum(valid_distances_km, 0.1)
                        valid_distances_10km = valid_distances_km / 10.0
                        proximity_exposure = np.sum(1.0 / valid_distances_10km)
                        df_coal.at[idx, col_name_hyper] = proximity_exposure
                    else:
                        df_coal.at[idx, col_name_hyper] = 0.0
                    
                    # Other type
                    col_name_other = f'ai_proximity_{distance_km}km_{window_name}_other'
                    valid_ai_other = valid_ai & (~ai_is_hyperscale)
                    
                    if valid_ai_other.sum() > 0:
                        valid_distances_km = distances_km[valid_ai_other]
                        valid_distances_km = np.maximum(valid_distances_km, 0.1)
                        valid_distances_10km = valid_distances_km / 10.0
                        proximity_exposure = np.sum(1.0 / valid_distances_10km)
                        df_coal.at[idx, col_name_other] = proximity_exposure
                    else:
                        df_coal.at[idx, col_name_other] = 0.0
            
            processed_count += 1
        
        # Progress report
        current_time = time.time()
        if processed_count % 1000 == 0 or (current_time - last_report_time) > 60:
            elapsed = current_time - start_time
            speed = processed_count / elapsed if elapsed > 0 else 0
            remaining = len(df_coal) - processed_count
            eta_seconds = remaining / speed if speed > 0 else 0
            
            print(f"\n{'='*80}")
            print(f"Progress report [{datetime.now().strftime('%H:%M:%S')}]")
            print(f"{'='*80}")
            print(f"Processed: {processed_count:,}/{len(df_coal):,} records ({processed_count/len(df_coal)*100:.1f}%)")
            print(f"Speed: {speed:.2f} records/sec")
            print(f"Elapsed: {elapsed/60:.1f} min")
            print(f"Remaining: {eta_seconds/60:.1f} min")
            print(f"ETA: {(datetime.now() + pd.Timedelta(seconds=eta_seconds)).strftime('%H:%M:%S')}")
            print(f"{'='*80}\n")
            
            last_report_time = current_time
    
    total_elapsed = time.time() - start_time
    print(f"\n✓ Calculation done! Total time: {total_elapsed/60:.1f} min")
    
    # ===== 6. Check consistency of type decomposition (all time windows) =====
    print("\n🔍 Step 6/6: Check type decomposition consistency...")
    print("="*80)
    
    for distance_km in buffer_distances:
        print(f"\n{distance_km}km buffer:")
        
        for window_name in time_windows.keys():
            col_total = f'ai_proximity_{distance_km}km_{window_name}'
            col_hyper = f'ai_proximity_{distance_km}km_{window_name}_hyper'
            col_other = f'ai_proximity_{distance_km}km_{window_name}_other'
            
            # Calculate total
            sum_parts = df_coal[col_hyper] + df_coal[col_other]
            
            # Check consistency
            diff = df_coal[col_total] - sum_parts
            max_diff = diff.abs().max()
            
            print(f"\n  Time window: {window_name}")
            print(f"    - Total exposure: mean={df_coal[col_total].mean():.6f}, max={df_coal[col_total].max():.6f}")
            print(f"    - Hyper:          mean={df_coal[col_hyper].mean():.6f}, max={df_coal[col_hyper].max():.6f}")
            print(f"    - Other:          mean={df_coal[col_other].mean():.6f}, max={df_coal[col_other].max():.6f}")
            print(f"    - Consistency:    max diff={max_diff:.10f} {'✓' if max_diff < 1e-6 else '❌'}")
            
            # Calculate shares
            total_mean = df_coal[col_total].mean()
            if total_mean > 0:
                hyper_pct = df_coal[col_hyper].mean() / total_mean * 100
                other_pct = df_coal[col_other].mean() / total_mean * 100
                print(f"    - Hyper share: {hyper_pct:.1f}%")
                print(f"    - Other share: {other_pct:.1f}%")
    
    # ===== 7. Data validation and statistics =====
    print("\n📈 Exposure statistics:")
    print("="*80)
    
    # Show statistics for all time windows
    for window_name in time_windows.keys():
        print(f"\n📊 {window_name} time window statistics:")
        for distance_km in buffer_distances:
            print(f"\n  {distance_km}km buffer:")
            
            for suffix in [window_name, f'{window_name}_hyper', f'{window_name}_other']:
                col_name = f'ai_proximity_{distance_km}km_{suffix}'
                col_data = df_coal[col_name]
                
                print(f"\n    {col_name}:")
                print(f"      - Mean: {col_data.mean():.6f}")
                print(f"      - Median: {col_data.median():.6f}")
                print(f"      - Std: {col_data.std():.6f}")
                print(f"      - Max: {col_data.max():.6f}")
                print(f"      - Non-zero records: {(col_data > 0).sum():,} ({(col_data > 0).sum()/len(df_coal)*100:.1f}%)")
    
    # ===== 8. Sample data preview =====
    print("\n📋 Sample data preview:")
    print("="*80)
    
    sample_cols = ['GEM_unit_phase_ID', 'year'] + [
        'ai_proximity_15km_before06',
        'ai_proximity_15km_before06_hyper',
        'ai_proximity_15km_before06_other',
        'ai_proximity_15km_20_24',
        'ai_proximity_15km_20_24_hyper',
        'ai_proximity_15km_20_24_other'
    ]
    print(df_coal[sample_cols].head(10).to_string())
    
    # ===== 9. Save results =====
    output_path = os.path.join(TEMP, "larger.dta")
    df_coal.to_stata(output_path, write_index=False, version=118)
    
    print(f"\n✅ Results saved to: {output_path}")
    print(f"✅ Total variables: {len(new_columns)}")
    print(f"  - Base variables: {base_vars}")
    print(f"  - Type decomposition variables: {type_vars} (all time windows)")
    
    return df_coal

def quick_check_results():
    """
    Quick check of the generated output file
    """
    output_path = os.path.join(TEMP, "larger.dta")
    
    try:
        df = pd.read_stata(output_path)
        
        print("\n🔍 Output file check:")
        print("="*80)
        
        # Check exposure variables
        proximity_vars = [col for col in df.columns if col.startswith('ai_proximity_')]
        
        # Category summary
        base_vars = [v for v in proximity_vars if not v.endswith('_hyper') and not v.endswith('_other')]
        type_vars = [v for v in proximity_vars if v.endswith('_hyper') or v.endswith('_other')]
        
        # Check GADM variables
        gadm_vars = [col for col in df.columns if col.startswith('GID_') or col.startswith('NAME_') or col.startswith('COUNTRY')]
        
        print(f"✓ Total records: {len(df):,}")
        print(f"✓ Total exposure variables: {len(proximity_vars)}")
        print(f"  - Base variables: {len(base_vars)}")
        print(f"  - Type decomposition variables: {len(type_vars)} (all time windows)")
        print(f"✓ GADM variables: {len(gadm_vars)}")
        
        print(f"\n✓ Base variable list:")
        for var in base_vars:
            print(f"  - {var}")
        
        print(f"\n✓ Type decomposition variable list:")
        for var in type_vars:
            print(f"  - {var}")
        
        # Statistical comparison (all time windows)
        print(f"\n📊 Variable comparison (all time windows):")
        print("="*80)
        
        time_windows = ['before06', '06_15', '16_19', '20_24']
        
        for window_name in time_windows:
            print(f"\n{window_name} time window:")
            for distance_km in [15, 25, 50]:
                print(f"\n  {distance_km}km buffer:")
                
                col_total = f'ai_proximity_{distance_km}km_{window_name}'
                col_hyper = f'ai_proximity_{distance_km}km_{window_name}_hyper'
                col_other = f'ai_proximity_{distance_km}km_{window_name}_other'
                
                print(f"    {col_total}: mean={df[col_total].mean():.6f}, non-zero={(df[col_total] > 0).sum():,}")
                print(f"    {col_hyper}: mean={df[col_hyper].mean():.6f}, non-zero={(df[col_hyper] > 0).sum():,}")
                print(f"    {col_other}: mean={df[col_other].mean():.6f}, non-zero={(df[col_other] > 0).sum():,}")
        
    except Exception as e:
        import traceback
        traceback.print_exc()

if __name__ == "__main__":
    try:
        result_df = process_proximity_exposure()
        quick_check_results()
        
    except KeyboardInterrupt:
        print("\n⚠️ Execution interrupted by user")
    except Exception as e:
        import traceback
        traceback.print_exc()
